# Crop Feature Selection for Agricultural Decision Support

## Project Overview

Soil measurements can help identify crops that are compatible with particular growing conditions. However, collecting multiple soil measurements may increase cost and effort.

This project investigates two related questions:

1. **If only one soil measurement can be used, which of nitrogen (N), phosphorus (P), potassium (K), or pH provides the strongest predictive signal for crop classification?**
2. **How much predictive performance is gained when all four soil measurements are used together?**

The analysis begins with a single-feature Logistic Regression baseline and extends it into a reproducible multi-feature machine-learning workflow using feature scaling, stratified cross-validation, and held-out test evaluation.

> **Portfolio extension:** The original exercise focused on identifying the strongest single predictive feature. This notebook extends that analysis by evaluating the predictive cost of the one-measurement constraint and validating a multi-feature model.


## 1. Setup and Data Loading

The dataset contains four numeric soil measurements and a categorical target variable, `crop`.

- `N` — nitrogen
- `P` — phosphorus
- `K` — potassium
- `ph` — soil pH
- `crop` — crop class to predict

The dataset file is intentionally not bundled with this repository. Place `soil_measures.csv` or `soil_measures.xlsx` in the `data/` folder (recommended) or upload it to the Colab session before running the analysis.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from pathlib import Path

# Place soil_measures.csv in data/, or upload it to the Colab session.
candidates = [
    Path("data/soil_measures.csv"),
    Path("/content/soil_measures.csv"),
    Path("soil_measures.csv"),
]
DATA_PATH = next((path for path in candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Place soil_measures.csv in data/ "
        "or upload it to the Colab session."
    )

crops = pd.read_csv(DATA_PATH)
expected_columns = {"N", "P", "K", "ph", "crop"}
if set(crops.columns) != expected_columns:
    raise ValueError(f"Expected columns {sorted(expected_columns)}, got {list(crops.columns)}")
print(f"Loaded {DATA_PATH} with {len(crops)} rows")
crops.head()


## 2. Dataset Overview

Before modeling, inspect the dataset structure, missing values, and target distribution. This provides a quick reproducibility check and confirms that the predictors are numeric and the target is multiclass.


In [ ]:
print("Dataset shape:", crops.shape)
print("\nColumn types:")
print(crops.dtypes)

print("\nMissing values:")
print(crops.isna().sum())

print("\nNumber of crop classes:", crops["crop"].nunique())
print("\nCrop class counts:")
print(crops["crop"].value_counts().sort_index())


## 3. Reproducible Train-Test Split

The predictors and target are separated and split into training and held-out test sets. A fixed `random_state` makes the split reproducible, while `stratify=y` preserves the crop-class distribution across both sets.


In [ ]:
X = crops.drop("crop", axis=1)
y = crops["crop"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 4. Single-Feature Baseline

A separate Logistic Regression model is trained for each soil measurement. Weighted F1 is used because this is a multiclass classification problem and the metric summarizes class-wise precision and recall while accounting for class frequency.

Each one-feature model uses a scaling-and-classification pipeline. This keeps the modeling procedure consistent and avoids optimization problems caused by feature scale.


In [ ]:
features = ["N", "P", "K", "ph"]
feature_performance = {}

for feature in features:
    single_feature_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ])

    single_feature_pipeline.fit(X_train[[feature]], y_train)
    y_pred = single_feature_pipeline.predict(X_test[[feature]])

    feature_performance[feature] = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

best_feature = max(feature_performance, key=feature_performance.get)
best_score = feature_performance[best_feature]
best_predictive_feature = {best_feature: best_score}

print("Single-feature weighted F1 scores:")
for feature, score in feature_performance.items():
    print(f"{feature}: {score:.3f}")

print(f"\nStrongest individual predictor: {best_feature} ({best_score:.3f})")


## 5. Multi-Feature Model

The portfolio extension asks what happens when the one-measurement constraint is removed. All four soil measurements are therefore combined in a single Logistic Regression pipeline.

`StandardScaler` and `LogisticRegression` are kept together in a pipeline so that scaling is learned only from the relevant training data.


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])


## 6. Stratified Cross-Validation

A single train-test split can be unusually easy or difficult. Five-fold stratified cross-validation provides a more robust estimate of the multi-feature model's performance.

The pipeline is evaluated inside each fold, preventing information from the validation fold from influencing feature scaling.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_weighted"
)

print("CV weighted F1 scores:", [round(score, 3) for score in cv_scores])
print(f"Mean CV weighted F1: {cv_scores.mean():.3f}")
print(f"CV standard deviation: {cv_scores.std():.3f}")


## 7. Final Held-Out Test Evaluation

After defining the modeling approach, the pipeline is trained on the complete training set and evaluated on the held-out test set.

Weighted F1 is the primary evaluation metric. Accuracy is also reported as an intuitive secondary measure of the proportion of correctly classified observations.


In [ ]:
pipeline.fit(X_train, y_train)
final_predictions = pipeline.predict(X_test)

final_f1 = f1_score(
    y_test,
    final_predictions,
    average="weighted"
)
final_accuracy = accuracy_score(y_test, final_predictions)

print(f"Final test weighted F1: {final_f1:.3f}")
print(f"Final test accuracy: {final_accuracy:.3f}")


## 8. Predictive Performance Comparison

The chart below compares the four one-measurement baselines with the final model using all available soil measurements. Because all bars use the same held-out test set and the same weighted F1 metric, the comparison directly illustrates the predictive cost of restricting the model to one soil measurement.


In [ ]:
comparison_scores = {
    "Nitrogen (N)": feature_performance["N"],
    "Phosphorus (P)": feature_performance["P"],
    "Potassium (K)": feature_performance["K"],
    "pH": feature_performance["ph"],
    "All Features": final_f1
}

plt.figure(figsize=(9, 5))
bars = plt.bar(comparison_scores.keys(), comparison_scores.values())

plt.ylabel("Weighted F1 Score")
plt.title("Crop Classification Performance by Available Soil Measurements")
plt.ylim(0, max(comparison_scores.values()) + 0.12)

for bar, score in zip(bars, comparison_scores.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"{score:.3f}",
        ha="center"
    )

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 9. Results and Interpretation

The single-feature experiment identifies the soil measurement that carries the strongest individual predictive signal for crop classification. However, the one-feature model remains substantially less informative than the model that combines all four soil measurements.

The multi-feature experiment therefore demonstrates the **predictive cost of the one-measurement constraint**: soil measurements contain complementary information, and combining them improves the model's ability to distinguish among crop classes.

The cross-validation results provide an additional check on model stability across different subsets of the training data, while the held-out test set provides the final evaluation.

These findings should be interpreted as **predictive relationships**, not causal effects. A feature being the strongest individual predictor does not mean that it alone determines crop suitability.


In [ ]:
print("PROJECT SUMMARY")
print("-" * 50)
print(f"Best single feature: {best_feature}")
print(f"Best single-feature weighted F1: {best_score:.3f}")
print(f"Mean 5-fold CV weighted F1 (all features): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Final test weighted F1 (all features): {final_f1:.3f}")
print(f"Final test accuracy (all features): {final_accuracy:.3f}")
print(f"Absolute F1 gain over best single feature: {final_f1 - best_score:.3f}")


## 10. Limitations and Future Work

This analysis is a baseline investigation and has several limitations:

- The results are based on one dataset and may not generalize to different geographical regions, soil conditions, climates, or farming practices.
- Only nitrogen, phosphorus, potassium, and pH are considered. Real-world crop suitability can also depend on rainfall, temperature, soil type, irrigation, and other environmental factors.
- Predictive performance does not establish a causal relationship between a soil measurement and crop suitability.
- Logistic Regression provides a useful baseline, but relationships between soil properties and crop classes may be nonlinear.
- The project compares predictive performance rather than the actual financial cost of soil testing.

Future work could compare nonlinear models, incorporate climatic and environmental variables, examine model explainability, and explicitly evaluate whether the predictive benefit of additional soil measurements justifies their real-world measurement cost.


## Conclusion

This project turns a simple feature-selection problem into a compact model-validation study. It demonstrates reproducible data splitting, multiclass evaluation with weighted F1, preprocessing through pipelines, stratified cross-validation, held-out evaluation, and interpretation of the trade-off between limited and richer feature availability.

The central practical finding is not that one soil property determines the correct crop, but that restricting the available information to a single measurement has a substantial predictive cost compared with combining multiple soil measurements.
